# One-touch 首达算子、自适应梯度与 Hepta DecisionCells

**研究核验日期：2026-10-30（America/New_York）**  
**范围：理论推导、固定源码检查、独立有限参数网格核验；未重跑仓库训练，未修改或激活生产系统。**

## 结论与身份

一触即付的数学核心是路径首达分布。它可以为概率预测、事件时间建模、价值传播及有明确偏差声明的替代梯度提供基础算子。原提案不能把平稳占据密度、首达概率导数和状态转移核互换，也不能从随机复位率推导自动节能。最有意义的研究对象是带首达、复位、不应期及任务价值的再生随机单元。

本笔记的公式推导是针对下文明确给定的神经随机过程，不声称 He–Xiong 或 Qi 已经证明这些神经网络结论。

本次读取的主干为 `732e934ba496d90fbe042b8d41d479ef901643e5`。直接相关研究为 adaptive 分支固定提交 `eb5a1cc503e1ea6592d68e73c95b94d24564352a`、distributional 分支固定提交 `4bcdecd2ff334b8705c565a5c77a4a97f05ba5e4`。代码中的 target 合同、研究实现与生产激活必须区分。

## 1. 文献实际支持的内容

He–Xiong (2012) 的资产是风险中性几何布朗运动；每笔债券期限为固定 m，到期时间均匀错开。Poisson 强度 ξ 表示债权人流动性冲击，ξk 进入债券有效贴现。对数变换使价值方程的齐次部分成为常系数二阶方程；完整股权值还含展期损益的非齐次项。原文最值得迁移的是延续价值、资源支出及内生停止边界的共同决定机制。该论文没有证明“OU 漏电等价于常数 killing/reset”。见 [作者原文](https://www.princeton.edu/~wxiong/papers/rollover.pdf)。

Qi 的《Deep Q-Learning on Hölder Spaces》确实存在。官方摘要定位为一致椭圆扩散中的 Bellman 目标正则性与逼近理论，状态方向平滑而动作方向仅 Lipschitz；不证明实际 sampled DQN 的训练收敛。本次取得全文为 arXiv:2606.16846v1（2026-06-15），不能声称已核验所有后续版本。Remark 2 使用全空间 Cauchy 问题避免吸收/反射边界问题。见 [PMLR](https://proceedings.mlr.press/v336/qi26a.html) 与 [原稿全文展示](https://www.researchgate.net/publication/407114790_Deep_Q-Learning_on_Holder_Spaces)。

## 2. 三种不同的“流失”机制

OU 漏电、Poisson 复位与终止 killing 的生成元分别为

$$\mathcal L_{OU}u=[I-a(x-r)]u_x+\tfrac12\sigma^2u_{xx},$$
$$\mathcal L_{reset}u=\mu u_x+\tfrac12\sigma^2u_{xx}+\lambda_r[u(r)-u(x)],$$
$$\mathcal L_{kill}u=\mu u_x+\tfrac12\sigma^2u_{xx}-\lambda_k u(x),$$

最后一个公式取 cemetery 的价值为零。若以完整状态空间表示，其跳跃项是 λ_k[u(†)-u(x)]，概率守恒由吸收状态承担。

复位与 OU 可以匹配一阶矩，但不能匹配整个过程：取 r=0，复位对 x² 的贡献为 −λ_r x²，OU 同强度漏电的贡献为 −2λ_r x²。二者首达分布自然不同。因此将漏电改成展期是一个可检验的新模型假设。

## 3. 单次合约的精确核

固定一个输入段，在实际神经过程的概率测度下，设

$$dX_t=\mu dt+\sigma dW_t,\quad X_0=x<b,\quad d=b-x>0,\quad \sigma>0,$$
$$\tau_b=\inf\{t\ge0:X_t\ge b\}.$$

reset r 是触界后的回跳点，默认不是下反射边界。以下解适用于 (−∞,b)；如另设下墙，必须重新求对应边值问题。

令 Φ、φ 为标准正态 CDF/PDF，h>0。利用吸收热核的镜像解得到

$$F(h,x)=\Pr_x(\tau_b\le h)=\Phi\!\left(\frac{\mu h-d}{\sigma\sqrt h}\right)
+e^{2\mu d/\sigma^2}\Phi\!\left(\frac{-\mu h-d}{\sigma\sqrt h}\right).$$

首达时间密度为

$$f(t,x)=\partial_tF=\frac{d}{\sigma\sqrt{2\pi t^3}}
\exp\!\left[-\frac{(d-\mu t)^2}{2\sigma^2t}\right].$$

生存概率 S=1−F。带非负 rate q 的首达量为

$$H_q(h,x)=\mathbb E_x[e^{-q\tau_b}\mathbf1_{\tau_b\le h}]=\int_0^h e^{-qt}f(t,x)dt.$$

令 κ=√(μ²+2qσ²)，则

$$H_q=e^{(\mu-\kappa)d/\sigma^2}\Phi\!\left(\frac{\kappa h-d}{\sigma\sqrt h}\right)
+e^{(\mu+\kappa)d/\sigma^2}\Phi\!\left(\frac{-\kappa h-d}{\sigma\sqrt h}\right).$$

证明：e^{-qt}f_μ(t)=e^{(μ−κ)d/σ²}f_κ(t)，积分后代入 F 即得。若 q=λ_k 是独立终止 hazard，H 是触界早于终止的概率；若 q=ρ 是效用贴现，H 是价值。两种含义不相同。两者同时存在时，条件期望中可出现 q=ρ+λ_k，但实际采样取消后不能再次重复扣同一取消概率。

真正的吸收转移核是

$$K_h(x,y)=\frac{e^{-(y-x-\mu h)^2/(2\sigma^2h)}}{\sigma\sqrt{2\pi h}}
\left[1-e^{-2(b-x)(b-y)/(\sigma^2h)}\right],\quad y<b.$$

它满足 K≥0、∫Kdy=S 和时间半群组合。K_h(x,dy) 的含义是仍未触界且末态位于 dy 的联合概率；条件生存核必须除以 S。带独立 killing 时未终止未触界核是 e^{-λ_k h}K_h。

独立终止使一个时间窗具有三个完备结果：

$$p_{hit}=H_{\lambda_k},\quad p_{survive}=e^{-\lambda_kh}S(h),$$
$$p_{cancel}=\int_0^h\lambda_ke^{-\lambda_kt}S(t)dt=1-p_{hit}-p_{survive}.$$

它们的参数 Jacobian、Hessian 求和分别为零。缺失观察不等于 survive。

## 4. 修复平稳密度与特征根

暂考虑 Poisson 复位率 λ_r>0（不是终止率），每次跳回 r；触界同样回 r，但触界后另有固定不应期 d_ref≥0。复位事件本身不增加不应期。活动质量 M 与发放率 ν 满足

$$0=\tfrac12\sigma^2p''-\mu p'-\lambda_rp+(\lambda_rM+\nu)\delta_r,$$
$$p(b)=0,\quad \nu=-\tfrac12\sigma^2p'(b^-),\quad M+\nu d_{ref}=1.$$

所以用户的特征方程确实对应前向稳态方程的齐次部分：

$$\gamma_\pm=\frac{\mu\pm\sqrt{\mu^2+2\lambda_r\sigma^2}}{\sigma^2}.$$

当 x<b 时，e^{γ_+(x−b)}−e^{γ_−(x−b)}<0。原写法的 D 必须取负。更透明的正写法，令 κ=√(μ²+2λ_rσ²)，是

$$p(x)=\frac{\nu}{\kappa}\begin{cases}
[e^{\gamma_-(r-b)}-e^{\gamma_+(r-b)}]e^{\gamma_+(x-r)},&x<r,\\
e^{\gamma_-(x-b)}-e^{\gamma_+(x-b)},&r\le x<b.
\end{cases}$$

它在 r 连续，导数跳跃满足 σ²[p'(r+)−p'(r−)]/2=−(λ_rM+ν)。遗漏 x<r 的部分或注入源，不能构成该再生过程的完整不变密度。

令 d=b−r，β=(κ−μ)/σ²。无复位首达时间的 Laplace 变换为 ψ(λ_r)=e^{-βd}。每次 attempt 成功概率为 ψ，平均 attempt 时长为 (1−ψ)/λ_r；更新方程给出

$$m_r=\frac{1-\psi}{\lambda_r\psi}=\frac{e^{\beta d}-1}{\lambda_r},\qquad
\nu=\frac1{d_{ref}+m_r}.$$

对上面的 p 积分直接得到 M=νm_r，故全部质量闭合。λ_r=0、μ>0 的极限为 m=d/μ；μ≤0 且不复位时不再有有限均值的平稳发放。

**非单调反例。** d=1、μ=1、σ=2 时，无 Poisson 复位的 m=1；λ_r=1 时 m=e^{1/2}−1≈0.648721，放电变快。加任意相同的正不应期不改变该方向。这与 [Evans–Majumdar 随机复位原始研究](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.106.160601) 的基本机制相容。

**时间尺度不可辨识性。** (μ,σ²,λ_r)→(cμ,cσ²,cλ_r) 不改变 γ_±。无固定不应期时 ν→cν 而 p 完全不变。只拟合归一化平稳形状，无法恢复绝对发放率或能耗。

## 5. 正确梯度与可学习参数

平稳占据密度 p(x) 与概率导数 ∂_xF 的单位相同，但不是同一个对象。尤其 p(b)=0，而 ∂_xF(b^-) 通常不为零。令 A=(μh−d)/(σ√h)、B=(−μh−d)/(σ√h)、R=e^{2μd/σ²}。用 φ(A)=Rφ(B) 得

$$g_F=\partial_xF=\frac{2\phi(A)}{\sigma\sqrt h}-\frac{2\mu}{\sigma^2}R\Phi(B),$$
$$\partial_\mu F=\frac{2d}{\sigma^2}R\Phi(B),\quad
\partial_\sigma F=\frac{2d\phi(A)}{\sigma^2\sqrt h}-\frac{4\mu d}{\sigma^3}R\Phi(B).$$

F 不包含取消率；物理取消影响的是 H_λ。有限 h 的 H 可通过闭式表达求导，也可稳定地微分积分：

$$\partial_\mu H_q=\int_0^h e^{-qt}f\frac{d-\mu t}{\sigma^2}dt,$$
$$\partial_\sigma H_q=\int_0^h e^{-qt}f\left[-\frac1\sigma+\frac{(d-\mu t)^2}{\sigma^3t}\right]dt,$$
$$\partial_qH_q=-\int_0^h t e^{-qt}fdt\le0.$$

在 d≥d_min>0、σ≥σ_min>0、有限参数紧集上，靠近 t=0 的 e^{-c/t} 控制所有多项式因子，支配收敛允许这些微分。H 的下降只表示一个合约的贴现/终止前支付概率下降，不表示再生后的总放电次数下降。

当 q>0、h→∞，H=e^{-βd}，β=(κ−μ)/σ²，满足 σ²β²/2+μβ−q=0。因此

$$\beta_\mu=-\beta/\kappa,\quad \beta_\sigma=-\sigma\beta^2/\kappa,\quad \beta_q=1/\kappa,$$
$$H_\mu=d\beta H/\kappa,\quad H_\sigma=d\sigma\beta^2H/\kappa,\quad H_q'=-dH/\kappa,$$
$$\partial_xH=\beta H.$$

最后一个可以作为具有明确首达意义的梯度候选。若它仅替代硬阶跃的 backward，则仍是 SG；若 H 本身参与连续前向目标，则它是那个连续目标的真导数。

特征根的链式导数（κ=√(μ²+2λσ²)、κ>0）为

$$\partial_\mu\gamma_\pm=(1\pm\mu/\kappa)/\sigma^2,$$
$$\partial_\sigma\gamma_\pm=\pm2\lambda/(\kappa\sigma)-2\gamma_\pm/\sigma,$$
$$\partial_\lambda\gamma_\pm=\pm1/\kappa.$$

若人为将指数差 A_θ(x) 归一化为 p_θ=A_θ/Z_θ，还须微分 Z：

$$\partial_\theta p_\theta=p_\theta[\partial_\theta\log A_\theta-
\mathbb E_{p_\theta}\partial_\theta\log A_\theta].$$

仅微分两个根而冻结幅度/归一化、重置分段或边界通常错误。

### 5.1 真正训练与元学习

把 θ 写成 `nn.Parameter` 不会凭空产生梯度。如果 θ 只出现在自定义 backward，而硬前向与 θ 无关，普通自动微分不会给出真实物理损失的 θ 梯度。

对于固定条件成本 c_j，三结果合约有一个明确的真目标：L(θ)=Σ_j p_j(θ)c_j，导数为 Σ_j c_j ∂p_j。条件成本本身若依赖 θ，需要再加 Σ_jp_j∂c_j。深层硬脉冲系统一般不满足 L(E[S])=E[L(S)]。可使用精确有限路径期望、合法的 likelihood-ratio estimator 或注明偏差的 SG/松弛。

若 θ 只训练 SG 形状，内层是 w_{k+1}=U_θ(w_k,z_k)，外层是验证集真实任务目标。离散路径枚举的完整超梯度为

$$\nabla_\theta J=\sum_\omega P_\theta(\omega)
\left[\nabla_\theta L_{val}(w_K(\omega,\theta))+
L_{val}(w_K(\omega,\theta))\nabla_\theta\log P_\theta(\omega)\right].$$

第一项包含对所有内层更新反传；第二项含路径概率改变。固定路径概率通常有偏。合法重参数化的路径导数已计入分布变化时，不能再重复加同一 score 项。

建议参数域通过投影或有界映射声明 μ∈[−M,M]、σ∈[σ_min,σ_max]、λ∈[0,Λ]，另有 reset gap≥d_min、不应期≥r_min、有限控制步与时间窗。这里的数值常数必须来自实际单位、噪声及事件率，不能冒充普适生理常数。物理参数与 SG 超参数使用不同名字和不同监督。

## 6. 可证明的稳定性与不能证明的收敛

1. **概率稳定。** K 非负且总质量≤1，故吸收传播在 sup 范数非扩张。把 spike/reset/refractory/cancel 全部分支纳入扩展状态后的 P_h 为完整 Markov 核。

2. **发放次数有界。** 如果每次 spike 后必须持有 d_ref≥r_min>0，则 N_T≤1+floor(T/r_min)，排除有限时间无限 spike。λ_k 只是 killing 时没有非平凡持续活动稳态；声称平稳必须另外声明再生/补充机制。

3. **Bellman 稳定。** 对固定时间步 h、折现 ρ>0、完整核 P_h 和有界奖励，TQ=r_h+e^{-ρh}P_h max_a Q 满足 ||TQ−TQ'||∞≤e^{-ρh}||Q−Q'||∞。精确值迭代收敛于其固定模型唯一不动点；每步 sup 误差≤ε 时，limsup 值误差≤ε/(1−e^{-ρh})。有限期限不需贴现，但局部误差随步数积累。此定理不包括任意函数逼近/离策略深度训练。

4. **优化稳定是额外命题。** 若实际标量目标 J 有 L-Lipschitz 梯度、下有界，且使用其精确梯度，则 η≤1/L 时 J(z−η∇J)≤J(z)−η||∇J||²/2。无偏、方差≤v² 的 SGD 对无投影平滑参数化给出平均梯度平方≤2(J_0−J_inf)/(ηN)+Lηv²；投影版本应以梯度映射表示。此处只得到驻点意义的结果，非深度网络全局最优。任意 SG 向量场不能代入该证明。

5. **元学习稳定。** 对有限 unroll 可直接追踪更新 Jacobian。若一个确定的内层更新映射满足 ||D_wU||≤c<1，则固定点敏感度为 (I−D_wU)^{-1}D_θU，范数≤||D_θU||/(1−c)。没有该稳定性条件就不能保证超梯度不爆炸。

6. **奇异边界必须保留。** μ=0 时，sup_{x<b}∂_xF=√(2/π)/(σ√h)。h↓0 或 σ↓0 导数上界发散。无限期限根公式在 μ=q=0 有退化；有限期限可用积分导数取正则极限。解析内核不意味着整个硬事件轨迹处处可微。

## 7. Hölder、smooth fit 与正确的伴随关系

真正的对偶来自 L=μ∂_x+σ²∂_{xx}/2 与 L*=−μ∂_x+σ²∂_{xx}/2。在边界条件及通量项处理正确时，∫p Lu=∫u L*p。前向密度、后向价值共享同一扩散半群。Hölder 是正则性概念，smooth fit 是特定自由边界最优性条件，二者不是一对对偶概念。

固定 one-touch 支付边界只要求 H_q(b^-)=1。以结算后常数 1 延拓时，左导数 β>0、右导数 0，通常不满足 smooth fit。内生干预边界则需任务价值：

$$\max\{\partial_tV+\mathcal LV+u-\rho V,\;\mathcal M_\pi V-V\}=0,$$
$$\mathcal M_\pi V=g_{spike}-\pi\cdot c_{spike}+\text{reset/refractory 后续值}.$$

在光滑非退化问题里，可有 V=MV 与 V_x=(MV)_x。任意复位、离散动作切换或预算边界不自动满足该条件。若每次发放只得常数正净奖励且允许自主立即发放，最优策略可能总是尽早发放；任务信息、延迟收益和实际成本才可能产生有意义的自由边界。

在连续时间 Spiking Q 架构中，首达核应进入 Bellman 目标：存活分支通过 K 传播，触界分支通过 f 积分 spike 收益与 reset 后继续价值，取消分支有自己的终值。不应期时钟进入状态；一个窗内允许多个 spike 时，递归/更新方程继续处理剩余时间，不能只算一次 F。

这样获得的是固定输入段、内部连续变量上的可微传播器。增广状态中的预算、时钟、离散相位不满足全空间一致椭圆；动作 max、到期角点与复位也不会因高斯核而全局解析。因此需要为这个混合系统重新证明边界和正则性，不能直接继承 Qi 的全空间定理。

### 7.1 一个真正可解的再生自由边界正例

动作电位仍是振幅1的事件；另行定义任务效用 g(x)=x−k（这不是原数字支付），每次 fire 后回 r<k，等待 τ>0，按 ρ>0 贴现。该特定无限期、常系数问题可精确求解。令 β>0 满足 σ²β²/2+μβ−ρ=0，D=e^{-ρτ}。延续区有 V(x)=A e^{βx}，价值匹配与 smooth fit 给出

$$V(b)=b-k+D V(r),\qquad \beta V(b)=1.$$

因此边界满足

$$G(b)=\beta(b-k)-1+D e^{-\beta(b-r)}=0,$$

且有唯一 b*∈(k,k+1/β)：G(k)<0、G(k+1/β)>0，区间内 G_b=β[1−D e^{-β(b−r)}]>0。值函数为

$$V_*(x)=\begin{cases}e^{\beta(x-b_*)}/\beta,&x<b_*,\\
x-b_*+1/\beta,&x\ge b_*.
\end{cases}$$

验证 QVI：延续区 LV−ρV=0，且 V−MV=[e^z−1−z]/β≥0，其中 z=β(x−b*)；干预区 MV=V，且 LV−ρV=μ−ρ(x−b*+1/β)≤−σ²β/2<0。加通常的折现可积性与正不应期，即为该标量问题的最优值验证。

若 θ 是 μ、σ、ρ 或其他边界参数，隐式导数为 ∂_θb*=−G_θ/G_b。保证 G_b≥g_min>0 才能得到统一边界敏感度界；唯一可解不等于数值条件数始终良好。这个模型也出现在仓库的再生研究中，但它不证明有输入耦合、有限预算或多层内容任务的全局最优性。

## 8. NDU 与 Organ Credit 的数学接口

Hepta 的一般递归效用 dU=−f(t,state,U,Z,a)dt+Z dM 不能一律改写成线性期望贴现奖励。在线性折现特例里可以直接用 F/H/K 构建 Bellman 目标；在一般递归 profile 下，解析扩散核只提供线性传播部分，非线性 aggregator、风险和噪声敏感性需保留。U 或 Z 都不是权重梯度。

将物理 hazard λ_phys（单位 1/time）与资源影子价格 π（单位 utility/resource）分开。对声明的折现线性 profile，可以求

$$\max_\theta J_{task}(\theta),\qquad C_j(\theta)\le B_j,$$
$$\mathcal J(\theta,\pi)=J_{task}(\theta)-\sum_j\pi_j(C_j(\theta)-B_j),\quad\pi_j\ge0.$$

价格根据已观察真实消耗更新：π_j^+=Π_[0,π_max](π_j+η(C_j−B_j))。若把 π 当普通 loss 参数共同下降，它可能退到零以移除惩罚，故方向不可混淆。

固定 continuation 的简单单交叉特例中，fire advantage=A(x)−πc，A'(x)>0、c>0，则 b*(π)=A^{-1}(πc)、db*/dπ=c/A'(b*)>0。这是一个确切的局部特化边界机制。一般递归价值依赖 π 时要重解问题；不能直接套这一导数。

更一般地，若每个 π 都精确求解相同固定模型的全局 J−πC，π_2>π_1 的两条最优性不等式相加得到 C(π_2)≤C(π_1)。近似 critic、局部优化、非平稳任务不保证该性质。对偶收敛还需要可行性、凸占用测度/恰当正则条件、有界噪声及合适步长。

Organ Credit 的价值归因与资源账本不同。主干候选 counterfactual advantage 为 A_i=Q_G(s,a)−Σ_b μ_i(b|o_i)Q_G(s,(a_-i,b))，只在对应条件/联合行为分布、critic 及支持集成立时有效，不能被当成任何 NDU 的通用梯度。实际资源只由执行 owner 结算一次；局部 cell 不创建新的奖励真相源。

## 9. Hepta 系统设计与伪代码

保留现有 `neuron.runtime` 状态 owner、`intuition.policy` 决策口、`utility.ndu` 估值和 `learning.*` 候选更新边界。cell 是逻辑状态单元，不为每个 cell 创建进程、数据库或完整 FBSDE solver。

建议新增一个版本化预测接口，字段为：

| 字段 | 语义 |
|---|---|
| p_hit / p_survive / p_cancel | 当前条件信息与指定 horizon 下的事件概率 |
| hit_time_law | 首达时间密度或有误差界的摘要 |
| survivor_distribution | 条件未触界的后验，不用均值冒充完整分布 |
| expected_count / expected_cost | 通过再生传播得到，不能设成一次 F |
| state_phase / remaining_refractory | 活动或不应期的持久状态 |
| model_revision / observation_frontier | 参数身份、实际条件信息和时间单位 |
| numerical_error / calibration_status | 计算误差与经验校准分别说明 |

以下是设计伪代码，方法名不是声称已存在的仓库 API：

```rust
fn decide(owner: &NeuronOwner, run: &Run, obs: &Observation) -> Receipt {
    let bundle = artifacts.selected_immutable_bundle(run.bundle_id);
    let state = owner.load_generation_bound_state(run.cell_id);
    let legal = guards.complete_legal_actions(run, &state, obs);

    let forecast = fpt.forecast(
        &state.belief, state.phase, state.remaining_refractory,
        &bundle.physical_parameters, obs,
        min(run.remaining_time, obs.constant_segment_remaining));
    forecast.validate_mass_units_and_domain()?;

    let values = ndu.evaluate_registered_profile(
        &legal, &forecast, &run.task_state, &run.resources,
        &run.resource_prices, &bundle.critic);
    let choice = policy.choose_and_record_actual_propensity(&legal, values);
    let receipt = owner.commit_choice_with_cas(
        state.generation, choice, forecast.identity());
    let outcome = taskflow.execute_or_reconcile_once(&receipt);
    ledger.settle_actual_usage_once(&receipt, &outcome);
    owner.apply_observed_successor(&receipt, &outcome);
    receipt
}

fn learn_next_bundle(data: FrozenDataset, selected: Bundle) -> Candidate {
    let mut candidate = selected.copy_registered_parameter_groups();
    let objective = data.fixed_objective_and_valid_estimator();
    train_with_true_event_or_declared_meta_gradient(&mut candidate, objective);
    trust_region_and_independent_evaluation(candidate)
}
```

Ready/预测分数不提供执行权限。先确定合法动作再估值，执行前仍按当前资源与状态原子预留/校验。参数在同一 run 的 selected bundle 内不可变，学习写 next snapshot 候选。静默且被观察到的未触界才更新条件生存分布；缺失观察另行处理。脉冲、开始动作、执行结果是不同事件，不得无依据地互相替代。

## 10. 固定源码证据与实际结果

本次读取固定提交，未复跑其训练。以下是仓库报告的范围内结果。

- [主干 DecisionCell 合同](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/732e934ba496d90fbe042b8d41d479ef901643e5/docs/learning/NEURAL_BIOMIMICRY_SPEC.md)：后端中性 target、既有 sparse kernel 与已训练 cell 区分、selected 参数不可变。
- [NDU 合同](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/732e934ba496d90fbe042b8d41d479ef901643e5/docs/learning/NDU_FBSDE_SPEC.md)：一般递归聚合与线性折现特例分开；U/Z 不是参数梯度。
- [Credit 合同](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/732e934ba496d90fbe042b8d41d479ef901643e5/docs/learning/CAUSAL_LONGITUDINAL_SPEC.md)：实际联合/条件行为概率、独立结果、反事实信用及资源归因限制。
- [Adaptive 合同](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/eb5a1cc503e1ea6592d68e73c95b94d24564352a/experiments/first_passage_adaptive/CONTRACT.md) 与 [状态](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/eb5a1cc503e1ea6592d68e73c95b94d24564352a/experiments/first_passage_adaptive/STATUS.json)：已实现三结果概率、全目标参数梯度、27 路径元梯度与研究桥接；生产激活 false。
- [冻结确认复核](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/eb5a1cc503e1ea6592d68e73c95b94d24564352a/experiments/first_passage_adaptive/review/CONFIRMATION_REVIEW.json)：fixed SG loss=0.3402562803、meta SG=0.3374559417、exact expected-gradient=0.2865819177；该有限已知律诊断不是等总计算成本或总体优势证明。物理训练 σ=0.2、λ=0 到下界；阈值 pooled 门通过而跨分层门失败。
- [信用反馈复核](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/eb5a1cc503e1ea6592d68e73c95b94d24564352a/experiments/first_passage_adaptive/review/CREDIT_REVIEW.json)：反馈使真实 read 从3降到1，但任务 return 从0.85降到−0.05；证明路径接通，不证明效用/能效提升。该旧复核中的恢复限制后来被 STATUS 部分更新，此处不将旧恢复缺口重复当作当前结论。
- [Distributional v2](https://github.com/TrillionniumFoundation/hepta-private-ci/blob/4bcdecd2ff334b8705c565a5c77a4a97f05ba5e4/experiments/hcsn_distributional_v2/README.md)：活动分布、不应期与联合训练已有研究实现；登记机制优势0/4，CPU完整 Adam 步约双门控18.12倍。仅说明该冻结配置的结果，不能证明所有首达架构必然无效。

相关 SNN 原始工作已讨论随机模型对 SG 的解释及深层偏差，故新颖性不能写成“首次给 SG 物理解释”。见 [Zenke Lab 作者说明及原论文入口](https://zenkelab.org/2024/04/elucidating-the-theoretical-underpinnings-of-surrogate-gradient-learning-in-spiking-neural-networks/)。合理的新贡献需要落在特定事件合同、联合训练、混合价值传播及同信息完整成本下的实际优势。

## 11. 下一步能区分机制的实验

以同一个硬结果任务和相同信息对比 true expected-gradient、固定 SG、学 shape 的 meta SG；物理参数学习与 SG shape 学习分组。把固定阈值、普通条件阈值、任务价值自由边界分开；同样提供资源、期限和状态。把 λ_k 取消、λ_r 复位、π 资源价格分别干预。显式计入完整训练/规划/推理、复位等待、事件率、内容质量及真实硬件测量。新科学结论使用新的研究登记与未用于旧选择的数据。

## 12. 本笔记附带的独立数值核验

下方代码不下载外部资料，不调用 GitHub，不运行训练，仅需 NumPy 与 SciPy。它检查24个 F 配置、48个 H 配置及24个平稳配置：时间积分、吸收核质量、三竞争事件质量、解析/积分梯度与中心差分、活动与不应期质量、时间尺度不变性、复位非单调反例。有限网格不能冒充全参数域浮点保证。结果已在本轮实际运行后保留在输出单元中。


In [1]:
"""Independent checks for the constant-coefficient one-touch research note.

NumPy/SciPy only. This checks a declared finite parameter grid, not a global
floating-point guarantee or a trained SNN. Run with Python 3.
"""
import itertools
import json
import math
from pathlib import Path

import numpy as np
from scipy.integrate import quad
from scipy.special import log_ndtr, ndtr


def first_passage_density(t, d, mu, sigma):
    if t <= 0:
        return 0.0
    log_f = (math.log(d) - math.log(sigma) - .5 * math.log(2 * math.pi)
             - 1.5 * math.log(t) - (d - mu * t)**2 / (2 * sigma**2 * t))
    return math.exp(log_f)


def hit_cdf(d, mu, sigma, horizon):
    scale = sigma * math.sqrt(horizon)
    aa, bb = (mu * horizon - d) / scale, (-mu * horizon - d) / scale
    return float(np.exp(np.logaddexp(log_ndtr(aa),
                                    2 * mu * d / sigma**2 + log_ndtr(bb))))


def discounted_hit(d, mu, sigma, rate, horizon):
    kappa = math.sqrt(mu**2 + 2 * rate * sigma**2)
    scale = sigma * math.sqrt(horizon)
    first = (mu - kappa) * d / sigma**2 + log_ndtr((kappa * horizon - d) / scale)
    second = (mu + kappa) * d / sigma**2 + log_ndtr((-kappa * horizon - d) / scale)
    return float(np.exp(np.logaddexp(first, second)))


def killed_kernel(y, d, mu, sigma, horizon):
    # x=0 and upper barrier b=d. Stable image difference, including drift.
    if y >= d:
        return 0.0
    density = math.exp(-(y - mu * horizon)**2 / (2 * sigma**2 * horizon))
    density /= sigma * math.sqrt(2 * math.pi * horizon)
    return density * (-math.expm1(-2 * d * (d - y) / (sigma**2 * horizon)))


def cdf_derivatives(d, mu, sigma, horizon):
    scale = sigma * math.sqrt(horizon)
    aa, bb = (mu * horizon - d) / scale, (-mu * horizon - d) / scale
    phi = math.exp(-aa**2 / 2) / math.sqrt(2 * math.pi)
    mirror_cdf = math.exp(2 * mu * d / sigma**2 + log_ndtr(bb))
    return np.array([
        -2 * phi / scale + 2 * mu * mirror_cdf / sigma**2,
        2 * d * mirror_cdf / sigma**2,
        2 * d * phi / (sigma**2 * math.sqrt(horizon))
        - 4 * mu * d * mirror_cdf / sigma**3,
        first_passage_density(horizon, d, mu, sigma),
    ])  # d, mu, sigma, horizon


def h_score_derivatives(d, mu, sigma, rate, horizon):
    def component(t, which):
        if t <= 0:
            return 0.0
        z = d - mu * t
        scores = (1 / d - z / (sigma**2 * t),
                  z / sigma**2,
                  -1 / sigma + z**2 / (sigma**3 * t), -t)
        return math.exp(-rate * t) * first_passage_density(t, d, mu, sigma) * scores[which]
    return np.array([quad(component, 0, horizon, args=(i,), epsabs=2e-11,
                          epsrel=2e-11, limit=250)[0] for i in range(4)])


def central_gradient(fn, values):
    values = np.array(values, dtype=float)
    out = []
    for i, val in enumerate(values):
        step = 2e-5 * max(1, abs(val))
        plus, minus = values.copy(), values.copy()
        plus[i] += step
        minus[i] -= step
        out.append((fn(*plus) - fn(*minus)) / (2 * step))
    return np.array(out)


def beta(drift, sigma, rate):
    kappa = math.sqrt(drift**2 + 2 * rate * sigma**2)
    # Avoid cancellation in kappa-drift for positive drift.
    return 2 * rate / (kappa + drift) if drift >= 0 and rate > 0 else (kappa - drift) / sigma**2


def restart_mean(d, mu, sigma, reset_rate):
    if reset_rate == 0:
        return d / mu if mu > 0 else math.inf
    return math.expm1(beta(mu, sigma, reset_rate) * d) / reset_rate


def stationary_density(v, d, mu, sigma, reset_rate, refractory):
    # reset=0, barrier=d; resets themselves do not cause refractory holding.
    kappa = math.sqrt(mu**2 + 2 * reset_rate * sigma**2)
    gp, gm = (mu + kappa) / sigma**2, (mu - kappa) / sigma**2
    firing_rate = 1 / (refractory + restart_mean(d, mu, sigma, reset_rate))
    if v >= d:
        return 0.0
    if v < 0:
        return (firing_rate / kappa * (math.exp(-gm * d) - math.exp(-gp * d))
                * math.exp(gp * v))
    return firing_rate / kappa * (math.exp(gm * (v-d)) - math.exp(gp * (v-d)))


def run_audit():
    errors = {k: 0.0 for k in (
        'cdf_vs_time_quadrature', 'discounted_hit_vs_time_quadrature',
        'kernel_plus_hit_mass', 'competing_events_mass',
        'cdf_gradient_abs_error', 'discounted_hit_gradient_abs_error',
        'stationary_plus_refractory_mass', 'stationary_time_rescaling')}
    grid = list(itertools.product([.25, 1.0], [-.7, 0.0, 1.0], [.5, 1.5], [.2, 1.3]))
    for d, mu, sigma, horizon in grid:
        f_int = quad(first_passage_density, 0, horizon, args=(d, mu, sigma),
                     epsabs=2e-11, epsrel=2e-11, limit=250)[0]
        ff = hit_cdf(d, mu, sigma, horizon)
        errors['cdf_vs_time_quadrature'] = max(errors['cdf_vs_time_quadrature'], abs(ff-f_int))
        kk = quad(killed_kernel, -np.inf, d, args=(d, mu, sigma, horizon),
                  epsabs=2e-11, epsrel=2e-11, limit=250)[0]
        errors['kernel_plus_hit_mass'] = max(errors['kernel_plus_hit_mass'], abs(kk+ff-1))
        grad = cdf_derivatives(d, mu, sigma, horizon)
        reference = central_gradient(hit_cdf, [d, mu, sigma, horizon])
        errors['cdf_gradient_abs_error'] = max(errors['cdf_gradient_abs_error'], float(np.max(abs(grad-reference))))
        for rate in [.2, 1.1]:
            hh = discounted_hit(d, mu, sigma, rate, horizon)
            h_int = quad(lambda t: math.exp(-rate*t)*first_passage_density(t,d,mu,sigma),
                         0, horizon, epsabs=2e-11, epsrel=2e-11, limit=250)[0]
            errors['discounted_hit_vs_time_quadrature'] = max(errors['discounted_hit_vs_time_quadrature'],abs(hh-h_int))
            cancellation = quad(lambda t: rate*math.exp(-rate*t)*(1-hit_cdf(d,mu,sigma,t))
                                if t>0 else rate, 0, horizon,
                                epsabs=2e-11, epsrel=2e-11, limit=250)[0]
            survival = math.exp(-rate*horizon)*(1-ff)
            errors['competing_events_mass'] = max(errors['competing_events_mass'],abs(hh+survival+cancellation-1))
            grad_h = h_score_derivatives(d, mu, sigma, rate, horizon)
            ref_h = central_gradient(lambda dd,mm,ss,rr: discounted_hit(dd,mm,ss,rr,horizon),
                                     [d, mu, sigma, rate])
            errors['discounted_hit_gradient_abs_error'] = max(errors['discounted_hit_gradient_abs_error'],
                                                             float(np.max(abs(grad_h-ref_h))))
    stationary_grid = list(itertools.product([-.5,.5,1.0],[.5,2.0],[.2,1.0],[0,.1]))
    for mu, sigma, rate, refractory in stationary_grid:
        low = quad(stationary_density, -np.inf, 0, args=(1,mu,sigma,rate,refractory),
                   epsabs=2e-11, epsrel=2e-11)[0]
        upper = quad(stationary_density, 0, 1, args=(1,mu,sigma,rate,refractory),
                     epsabs=2e-11, epsrel=2e-11)[0]
        nu = 1/(refractory+restart_mean(1,mu,sigma,rate))
        errors['stationary_plus_refractory_mass'] = max(errors['stationary_plus_refractory_mass'],
                                                       abs(low+upper+nu*refractory-1))
    # Same stationary shape under acceleration of time, without fixed refractory delay.
    speed = 3.0
    for v in np.linspace(-3,.999,200):
        p = stationary_density(v,1,.6,1.2,.4,0)
        accelerated = stationary_density(v,1,speed*.6,math.sqrt(speed)*1.2,speed*.4,0)
        errors['stationary_time_rescaling'] = max(errors['stationary_time_rescaling'],abs(p-accelerated))
    counterexample = []
    for rate in [0,1,5,10,100]:
        mean = restart_mean(1,1,2,rate)
        counterexample.append({'reset_rate':rate, 'mean_active_first_passage':mean,
                              'spike_rate_with_refractory_0_1':1/(mean+.1)})
    limits = {key: (2e-7 if 'gradient' in key else 1e-8) for key in errors}
    assert all(errors[k] < limits[k] for k in errors), errors
    return {'scope':'Independent finite-grid analytic consistency checks; no repo training rerun',
            'cdf_parameter_cases':len(grid), 'discounted_cases':2*len(grid),
            'stationary_cases':len(stationary_grid),
            'maximum_absolute_errors':errors, 'reset_counterexample':counterexample,
            'all_declared_checks_passed':True,
            'limitations':['Finite grid only; not an exhaustive extreme-tail or float-error certificate.',
                           'No Monte Carlo or event-path gradient claim.',
                           'No production Hepta activation or general training/energy claim.']}


AUDIT_RESULT = run_audit()
print(json.dumps(AUDIT_RESULT, indent=2, ensure_ascii=False))

{
  "scope": "Independent finite-grid analytic consistency checks; no repo training rerun",
  "cdf_parameter_cases": 24,
  "discounted_cases": 48,
  "stationary_cases": 24,
  "maximum_absolute_errors": {
    "cdf_vs_time_quadrature": 1.6653345369377348e-15,
    "discounted_hit_vs_time_quadrature": 2.275957200481571e-15,
    "kernel_plus_hit_mass": 2.220446049250313e-16,
    "competing_events_mass": 7.771561172376096e-16,
    "cdf_gradient_abs_error": 6.803189123161246e-09,
    "discounted_hit_gradient_abs_error": 5.257176471928915e-09,
    "stationary_plus_refractory_mass": 4.440892098500626e-16,
    "stationary_time_rescaling": 5.551115123125783e-16
  },
  "reset_counterexample": [
    {
      "reset_rate": 0,
      "mean_active_first_passage": 1.0,
      "spike_rate_with_refractory_0_1": 0.9090909090909091
    },
    {
      "reset_rate": 1,
      "mean_active_first_passage": 0.6487212707001282,
      "spike_rate_with_refractory_0_1": 1.3356105123938866
    },
    {
      "reset_rate